# TimesFM - Google Pretrained Foundation Model (BONUS)

MLflow experiment: `TimesFM_Training`.
Runs: `TimesFM_ZeroShot`, `TimesFM_Analysis`.

Google's pretrained TimesFM used **zero-shot** - no fine-tuning, no exogenous inputs - on the same
39-week horizon as every other notebook. Metric: WMAE, holiday weeks weighted 5x.

Two scoping decisions, both deliberate:

**Subset, not the full panel.** Zero-shot forecasting all ~3,000 series is slow and adds nothing to
the argument, so this evaluates the same `N_SUBSET` high-volume series the Prophet notebook uses -
identical series, so the two are directly comparable.

**A seasonal-naive baseline is scored alongside it.** The question a foundation-model track exists to
answer is not "what WMAE does TimesFM get" - that number is meaningless alone. It is *"does a general
pretrained model beat the cheapest forecast available?"* The cheapest forecast here is lag-52: the
same week one year ago. Given that XGBoost's single strongest feature is exactly that signal, the bar
is high, and the comparison is the finding.

## Environment

In [ ]:
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import os
    os.chdir("/content")                            # never nest the clone on a re-run
    if os.path.isdir("/content/ML-final"):
        !git -C /content/ML-final pull --ff-only    # a warm runtime must not keep stale modules
    else:
        !git clone https://github.com/ketevan614/ML-final.git
    %cd /content/ML-final

    !pip -q install mlflow dagshub kaggle pandas pyarrow
    !pip -q install git+https://github.com/google-research/timesfm.git

    from google.colab import userdata
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        f.write(userdata.get("KAGGLE_JSON"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

    if not os.path.exists("data/train.csv"):
        !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
        import zipfile, glob
        os.chdir("data")
        while glob.glob("*.zip"):
            for z in glob.glob("*.zip"):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(".")
                os.remove(z)
        os.chdir("/content/ML-final")

print("IN_COLAB =", IN_COLAB)

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

def find_data_dir(root):
    """The Kaggle zip sometimes unpacks into a nested folder; accept either layout."""
    required = {"train.csv", "test.csv", "features.csv", "stores.csv", "sampleSubmission.csv"}
    for d in [root / "data", root / "data" / "walmart-recruiting-store-sales-forecasting"]:
        if d.exists() and required.issubset({p.name for p in d.glob("*.csv")}):
            return d
    raise FileNotFoundError("Could not find all 5 CSVs in data/ or the nested Kaggle folder.")

DATA_DIR = find_data_dir(ROOT)
print("ROOT =", ROOT, "| DATA_DIR =", DATA_DIR)

## MLflow / DagsHub tracking

In [ ]:
import os
import mlflow

for var in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"):
    os.environ.pop(var, None)

if IN_COLAB:
    from google.colab import userdata
    os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")

os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/karev23/ML-final.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "gabas22"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("TimesFM_Training")
print("tracking:", mlflow.get_tracking_uri())

def safe_log_params(params):
    try:
        mlflow.log_params(params)
    except Exception as e:
        print("mlflow.log_params failed:", e)

def safe_log_param(key, value):
    try:
        mlflow.log_param(key, value)
    except Exception as e:
        print(f"mlflow.log_param({key}) failed:", e)

def safe_log_metric(key, value):
    try:
        mlflow.log_metric(key, value)
    except Exception as e:
        print(f"mlflow.log_metric({key}) failed:", e)

def safe_log_artifact(path):
    try:
        mlflow.log_artifact(path)
    except Exception as e:
        print(f"mlflow.log_artifact({path}) failed:", e)

## Imports and data

In [ ]:
import numpy as np
import pandas as pd

from dataloader import load_raw
from metrics import wmae
from validation import holdout_split

try:
    import timesfm
    TIMESFM_AVAILABLE = True
except ImportError:
    TIMESFM_AVAILABLE = False
    print("timesfm not installed - the zero-shot cell will be skipped, the baseline still runs.")

train, test, features, stores, sample = load_raw(DATA_DIR)
HORIZON = 39
N_SUBSET = 5          # same as the Prophet notebook, so both score identical series

print("train:", train.shape, "| timesfm available:", TIMESFM_AVAILABLE)

## Subset selection

Top `N_SUBSET` series by total historical sales - the same rule and the same N as the Prophet
notebook, so the two bonus/theory tracks are scored on identical series.

In [ ]:
series_totals = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"].sum().sort_values(ascending=False)
)
subset_keys = series_totals.head(N_SUBSET).index.tolist()

def subset_folds():
    """Yield (store, dept, train_values, val_frame) - last HORIZON weeks held out."""
    for store, dept in subset_keys:
        s = train[(train["Store"] == store) & (train["Dept"] == dept)].sort_values("Date")
        tr_mask, va_mask = holdout_split(s["Date"].to_numpy(), horizon=HORIZON)
        tr, va = s[tr_mask], s[va_mask]
        if len(tr) < HORIZON or len(va) == 0:
            continue
        yield store, dept, tr["Weekly_Sales"].to_numpy(dtype=float), va

for store, dept in subset_keys:
    print(f"  Store {store}, Dept {dept}  (total sales {series_totals[(store, dept)]:,.0f})")

## Baseline: seasonal-naive (lag-52)

"Predict the same week last year." No parameters, no training, no data beyond the series itself. This
is the bar TimesFM has to clear to justify its checkpoint download - and XGBoost's feature
importances say the bar is high, because `sales_lag_52` and `store_dept_history_mean` are precisely
the features carrying that model.

In [ ]:
def seasonal_naive(tr_values, horizon):
    if len(tr_values) >= 52:
        base = tr_values[-52:]
        reps = int(np.ceil(horizon / 52))
        return np.clip(np.tile(base, reps)[:horizon], 0, None)
    return np.clip(np.full(horizon, float(np.mean(tr_values))), 0, None)

naive_scores = []
for store, dept, tr_values, va in subset_folds():
    p = seasonal_naive(tr_values, len(va))
    naive_scores.append(
        wmae(va["Weekly_Sales"].to_numpy(), p, va["IsHoliday"].astype(bool).to_numpy())
    )

mean_naive = float(np.mean(naive_scores)) if naive_scores else float("nan")
print("seasonal-naive (lag-52) subset mean WMAE:", round(mean_naive, 1))

## Run: TimesFM_ZeroShot

Feed each series' pre-holdout history to the pretrained checkpoint as context and forecast 39 weeks
ahead. No fine-tuning, no exogenous features - TimesFM's zero-shot interface is univariate.

`load_timesfm` handles both published APIs: **2.5** (`from_pretrained` + `compile(ForecastConfig)`,
which infers the sampling rate itself) and **1.x** (`TimesFmHparams` + an explicit `freq` bucket,
where weekly data is bucket `1`, not `0`).

In [ ]:
def load_timesfm(horizon, max_context=512):
    """Return (forecast_fn, checkpoint_name), supporting the 2.5 and 1.x APIs."""
    if hasattr(timesfm, "TimesFM_2p5_200M_torch"):
        name = "google/timesfm-2.5-200m-pytorch"
        m = timesfm.TimesFM_2p5_200M_torch.from_pretrained(name)
        m.compile(
            timesfm.ForecastConfig(
                max_context=max_context,
                max_horizon=horizon,
                normalize_inputs=True,
                infer_is_positive=True,
            )
        )

        def forecast(series):
            point, _ = m.forecast(horizon=horizon, inputs=[np.asarray(series, dtype=float)])
            return np.asarray(point[0][:horizon], dtype=float)

        return forecast, name

    name = "google/timesfm-1.0-200m-pytorch"
    m = timesfm.TimesFm(
        hparams=timesfm.TimesFmHparams(backend="cpu", per_core_batch_size=32,
                                       horizon_len=horizon),
        checkpoint=timesfm.TimesFmCheckpoint(huggingface_repo_id=name),
    )

    def forecast(series):
        # freq bucket 1 = weekly/monthly. Bucket 0 would claim the data is daily.
        point, _ = m.forecast([np.asarray(series, dtype=float)], freq=[1])
        return np.asarray(point[0][:horizon], dtype=float)

    return forecast, name


zero_shot_scores = []
mean_zero_shot = float("nan")

if TIMESFM_AVAILABLE:
    forecast_fn, checkpoint = load_timesfm(HORIZON)
    print("checkpoint:", checkpoint)

    with mlflow.start_run(run_name="TimesFM_ZeroShot"):
        safe_log_param("checkpoint", checkpoint)
        safe_log_param("n_subset", N_SUBSET)
        safe_log_param("fine_tuned", False)
        safe_log_param("uses_exogenous", False)

        for i, (store, dept, tr_values, va) in enumerate(subset_folds()):
            p = np.clip(forecast_fn(tr_values)[:len(va)], 0, None)
            score = wmae(va["Weekly_Sales"].to_numpy(), p,
                         va["IsHoliday"].astype(bool).to_numpy())
            zero_shot_scores.append(score)
            safe_log_metric(f"wmae_series_{i}", score)
            print(f"Store {store} Dept {dept}:  TimesFM {score:>10,.1f}   naive {naive_scores[i]:>10,.1f}")

        mean_zero_shot = float(np.mean(zero_shot_scores)) if zero_shot_scores else float("nan")
        safe_log_metric("wmae_subset_mean", mean_zero_shot)
        safe_log_metric("wmae_subset_mean_seasonal_naive", mean_naive)
        print(f"\nsubset mean WMAE   TimesFM {mean_zero_shot:,.1f}   seasonal-naive {mean_naive:,.1f}")
else:
    print("Skipped: timesfm not installed.")

BEATS_NAIVE = bool(zero_shot_scores) and mean_zero_shot < mean_naive
print("TimesFM beats seasonal-naive:", BEATS_NAIVE)

## Run: TimesFM_Analysis

Log the zero-shot result against the baseline, plus the written analysis, as an artifact for the
README's bonus section.

In [ ]:
ANALYSIS = f"""TimesFM zero-shot analysis
==========================

Subset mean WMAE, zero-shot (no fine-tuning): {mean_zero_shot:,.1f}
Subset mean WMAE, seasonal-naive (lag-52):    {mean_naive:,.1f}
Beats the naive baseline:                     {"yes" if BEATS_NAIVE else "no"}

A pretrained foundation model arrives knowing nothing about this dataset. Ranked by how
much each gap costs under WMAE:

1. No holiday awareness - and WMAE weights holiday weeks 5x. TimesFM cannot know a week
   is Thanksgiving or Christmas. Those are the weeks where Walmart sales spike hardest
   and the metric punishes hardest. build_features hands the tree models is_thanksgiving,
   is_christmas and DaysToChristmas outright.
2. No exogenous signal. The zero-shot interface is univariate: no Temperature, Fuel_Price,
   CPI, Unemployment, no MarkDown promo columns.
3. No dataset-specific fitting. Pretrained on a broad generic corpus, not Walmart retail,
   so it cannot exploit this panel's structure - per-store scale, per-department behaviour,
   the strong year-over-year seasonality a single lag-52 column captures almost for free.

The seasonal-naive comparison is the point of this track. Lag-52 costs nothing and encodes
the single strongest regularity in the data. XGBoost's top feature importances are
dominated by exactly that signal (store_dept_history_mean 0.274, sales_lag_52), which is a
good reason to expect a general pretrained model to struggle to beat it here.

Where a foundation model still earns its place: zero training cost, a sane forecast for a
brand-new series with no history to fit on, and as one more input to an ensemble on top of
the tuned models.
"""

with open("timesfm_analysis.txt", "w") as f:
    f.write(ANALYSIS)

with mlflow.start_run(run_name="TimesFM_Analysis"):
    safe_log_metric("wmae_subset_mean", mean_zero_shot)
    safe_log_metric("wmae_subset_mean_seasonal_naive", mean_naive)
    safe_log_param("beats_seasonal_naive", BEATS_NAIVE)
    safe_log_artifact("timesfm_analysis.txt")

print(ANALYSIS)

## Results

- Subset zero-shot WMAE: ___  | seasonal-naive (lag-52), same series and weeks: ___
- Did TimesFM beat the naive baseline: ___
- vs Prophet on the same 5 series: ___
- vs XGBoost / PatchTST: ___

Note on comparability: this is a **5-series subset**, so its WMAE is not on the same scale as the
full-panel numbers (XGBoost CV 2,755.4, PatchTST holdout 2,581.9). The number that means something
here is the **ratio to the seasonal-naive baseline on the same series**, not the absolute value.

- Takeaway for the README's bonus section: ___